# 02 — Test predictions and analyse interventions

Prerequisites, in order: hypotheses frozen; test sets written and locked; blind sheets labelled; then
`python scripts/evaluate.py` and `python scripts/run_interventions.py`.

**Predictive evidence** (does the hypothesis forecast activations on unseen inputs?) and **causal evidence** (does editing the neuron change the output?) are shown separately, and never merged into one score. A causal effect does not establish what a neuron means.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import HTML, display
from transformers import AutoTokenizer
from neurogen.config import load_config
from neurogen import pipeline as P, viz

CONFIG = ROOT / "configs" / "qwen3_4b.yaml"   # <- change to the config you ran
cfg = load_config(CONFIG, root=str(ROOT))
OUT = P.run_dir(cfg)
HYP_DIR = ROOT / "research" / "hypotheses"
print("run dir:", OUT)

In [ ]:
from neurogen.hypotheses import verify_frozen
cands = {c["cid"]: c for c in P.load_candidates(OUT)}
subset = json.loads((OUT / "discovery" / "phase3_subset.json").read_text())["cids"]
for cid in subset:
    try:
        verify_frozen(HYP_DIR, cid); status = "frozen"
    except PermissionError as e:
        status = f"NOT READY: {e}"
    print(cid, status)

## 1. Predictive tests: summary
Every subset neuron is listed, including unevaluated ones.

In [ ]:
summary = pd.read_csv(OUT / "evaluation" / "summary.csv")
summary

In [ ]:
def load_eval(cid):
    p = OUT / "evaluation" / f"{cid}.json"
    return json.loads(p.read_text()) if p.exists() else None

CID = subset[0]     # <- change
ev = load_eval(CID)
if ev:
    for part in ("authored", "blind", "combined"):
        r = ev[part]
        if not r: continue
        print(f"--- {part}: verdict = {r['verdict']}  ({r['reason']})")
        print(pd.DataFrame({h: {k: r[h][k] for k in ("n", "coverage", "balanced_accuracy", "auc",
                                                      "counterexample_false_positive_rate")}
                            for h in ("h1", "h2")}).T.round(3).to_string())
        print("  95% CI bal.acc  H1", np.round(r["h1"]["balanced_accuracy_ci"], 2),
              " H2", np.round(r["h2"]["balanced_accuracy_ci"], 2))
        print("  discrimination:", r["discrimination"])

In [ ]:
if ev:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    viz.plot_predictions(ev["items"], ev["threshold"], ax=axes[0], title=f"{CID}: authored items")
    if ev["blind_items"]:
        viz.plot_predictions(ev["blind_items"], ev["threshold"], ax=axes[1], title="blind held-out items")
    plt.tight_layout(); plt.show()
    display(pd.DataFrame(ev["items"])[["id", "category", "h1", "h2", "target_act", "max_act_anywhere", "argmax_token"]])

### Held-out activation distribution by genre
Now allowed: the hypotheses are frozen.

In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.model.name, revision=cfg.model.revision)
disc, held = P.load_corpora(cfg, tok)
ht = np.load(OUT / "heldout_SEALED" / "traces.npz")
dt = np.load(OUT / "discovery" / "traces.npz")
c = cands[CID]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
viz.plot_genre_distribution(dt[CID], P.genre_array(disc), c["polarity"], c["active_threshold"], ax=axes[0], title="discovery")
viz.plot_genre_distribution(ht[CID], P.genre_array(held), c["polarity"], c["active_threshold"], ax=axes[1], title="held-out")
plt.tight_layout(); plt.show()

### Exploratory probing (does NOT count as evidence)
Try sentences here to build intuition. Any sentence you probe here must **not** be added to a test set afterwards.

In [ ]:
RUN_MODEL = False   # loading the model takes memory; enable when needed
if RUN_MODEL:
    from neurogen.model import load_model
    from neurogen.predictions import TestItem, measure_items
    lm = load_model(cfg.model, batch_size=1, seq_len=128)
    probe = TestItem("probe", "Type a sentence here.", "sentence", "negative", "inactive", "inactive")
    m = measure_items(lm.model, lm.tokenizer, c["layer"], c["neuron"], c["polarity"], [probe], lm.device)[0]
    print(m["target_act"], list(zip(lm.tokenizer.tokenize(probe.text), np.round(m["token_acts"], 2))))

## 2. Causal interventions

In [ ]:
def load_int(cid):
    p = OUT / "interventions" / f"{cid}.json"
    return json.loads(p.read_text()) if p.exists() else None

iv = load_int(CID)
if iv:
    ctx = iv["contexts"]
    df = pd.DataFrame([dict(example=x["example_id"], genre=x["genre"], h1_feature=x["h1_feature"],
                            act=x["activation"], next_tok=x["true_next_token"],
                            zero_KL=x["zero_ablation"]["kl_mean"],
                            pct_vs_controls=x["zero_ablation"]["percentile_vs_controls"],
                            z=x["zero_ablation"]["z_vs_controls"],
                            mean_abl_KL=x["mean_ablation"]["kl_mean"],
                            d_logp_next=(list(x["zero_ablation"]["delta_logprob"].values()) or [[np.nan]])[0][0])
                       for x in ctx])
    display(df.round(4))
    fig, axes = plt.subplots(1, 3, figsize=(15, 3))
    viz.plot_control_comparison(ctx[0]["zero_ablation"], ax=axes[0], title="zero-ablation, strongest context")
    dr = sorted(iv["dose_response"], key=lambda d: d["factor"])
    viz.plot_dose_response(dr, ax=axes[1])
    S = np.array(iv["effect_similarity"])
    im = axes[2].imshow(S, vmin=-1, vmax=1, cmap="RdBu_r")
    labels = [f"{x['example_id']}:{x['h1_feature'] or '-'}" for x in ctx]
    axes[2].set_xticks(range(len(labels)), labels, rotation=60, ha="right", fontsize=7)
    axes[2].set_yticks(range(len(labels)), labels, fontsize=7)
    axes[2].set_title("cosine sim. of ablation effects (vocab)", loc="left"); axes[2].grid(False)
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    plt.tight_layout(); plt.show()
    print("direct-logit (approx., descriptive) promoted:", iv["direct_logit_effect"]["promoted_tokens"][:10])
    print("suppressed:", iv["direct_logit_effect"]["suppressed_tokens"][:10])

**Reading the similarity matrix.** Compare blocks for contexts H1 assigns to *different* features. Similar effects across features are compatible with a shared downstream role. Dissimilar effects mean downstream layers use the neuron's fixed output direction differently by context. Neither pattern settles H1 vs H2.

## 3. Summary table: predictive and causal evidence side by side

In [ ]:
rows = []
for cid in subset:
    s = summary[summary.cid == cid].iloc[0].to_dict()
    iv = load_int(cid)
    pct = [x["zero_ablation"]["percentile_vs_controls"] for x in iv["contexts"]] if iv else []
    hyp = HYP_DIR / f"{cid}.json"
    prior = json.loads(hyp.read_text())["prior"] if hyp.exists() else {}
    rows.append(dict(cid=cid, selection=cands[cid]["selection"],
                     predictive_verdict=s["verdict"], reason=s["reason"],
                     prior_h1=prior.get("h1"), prior_h2=prior.get("h2"), prior_neither=prior.get("neither"),
                     causal_median_pct_vs_controls=np.median(pct) if pct else np.nan,
                     causal_frac_contexts_above_95pct=np.mean(np.array(pct) >= 95) if pct else np.nan))
final = pd.DataFrame(rows)
final.to_csv(OUT / "final_summary.csv", index=False)
final

In [ ]:
# Verdict counts by how the neuron was selected. If random baselines get "supports H2"
# as often as heterogeneity-selected neurons, unified stories are cheap.
pd.crosstab(final.selection, final.predictive_verdict, margins=True)

## Interpretation checklist (copy into the write-up)
- [ ] All subset neurons reported, including "not evaluated" and random baselines.
- [ ] Verdicts come from the pre-registered rule; deviations are logged.
- [ ] Causal results are discussed separately from predictive verdicts.
- [ ] "Inconclusive (both fit)" is discussed as consistent with non-exclusive hypotheses.
- [ ] No claim about the share of interpretable neurons in the model.
- [ ] Limitations from `PROTOCOL.md` §7 are stated.